# Adaptive Step Size and Online Calibration

Two halves:

- **Part 1** uses the logistic ODE -- a smooth, easy problem -- to demonstrate **post-hoc calibration** and the relationship between the per-step quasi-MLE and the global MLE.
- **Part 2** uses a **two-component logistic with staggered transitions** -- a smooth problem with two well-separated fast regions -- to demonstrate **adaptive step control**. The per-step quasi-MLE feeds both the local error estimate (driving accept/reject) and the running covariance rescaling.


In [ ]:
import jax
import jax.numpy as np
import jax.scipy.stats as jss
import matplotlib.pyplot as plt
import numpy as onp

from ode_filters.calibration import (
    posthoc_mle_sigma_sqr,
    quasi_mle_sigma_sqr,
    rescale_sqr_seq,
)
from ode_filters.filters import (
    ekf1_sqr_adaptive_loop,
    ekf1_sqr_loop,
    ekf1_sqr_loop_dynamic,
)
from ode_filters.measurement import ODEInformation
from ode_filters.priors import IWP, taylor_mode_initialization

jax.config.update("jax_enable_x64", True)


## Part 1 -- Post-hoc calibration on the logistic ODE

$$\dot x = x(1-x), \qquad x(0)=0.1, \qquad t \in [0, 8].$$

Run a fixed-step EKF with `sigma = 1` (uncalibrated), then calibrate post-hoc.

### What "post-hoc MLE" optimises

Both calibrators come from the *same* Gaussian likelihood. At step $n$ the predicted-observation marginal is $(m_z^{(n)}, S_n)$; rescaling the prior diffusion by $\sigma^2$ turns $S_n$ into $\sigma^2 S_n$. The joint log-likelihood across $N$ steps is

$$\log p(m_z^{(1{:}N)}\mid \sigma^2) = -\tfrac{1}{2}\sum_n \Big[ d\log(2\pi) + \log\!\big|\sigma^2 S_n\big| + \tfrac{1}{\sigma^2}\, m_z^{(n)\top} S_n^{-1} m_z^{(n)}\Big].$$

- Maximising **with one $\sigma_n^2$ per step** gives the per-step quasi-MLE $\widehat\sigma_n^2 = \tfrac{1}{d} m_z^{(n)\top} S_n^{-1} m_z^{(n)}$.
- Maximising **with one global $\sigma^2$ under the constant-sigma assumption** gives the post-hoc MLE $\widehat\sigma^2 = \tfrac{1}{Nd}\sum_n m_z^{(n)\top} S_n^{-1} m_z^{(n)}$.

Same likelihood, one-parameter constraint vs N-parameter. The closed forms give us $\widehat\sigma^2_{\text{post-hoc}} = \tfrac{1}{N}\sum_n \widehat\sigma^2_n$ -- the global MLE is the arithmetic mean of the per-step quasi-MLEs. We verify this numerically below.

In [ ]:
def vf_log(x, *, t):
    return x * (1 - x)


x0_log = np.array([0.1])
tspan_log = (0.0, 8.0)
prior_log = IWP(q=2, d=1)
mu0_log, S0_log = taylor_mode_initialization(vf_log, x0_log, q=2)
measure_log = ODEInformation(vf_log, prior_log.E0, prior_log.E1)

N_log = 10
res_log = ekf1_sqr_loop(mu0_log, S0_log, prior_log, measure_log, tspan_log, N_log)
m_log = np.stack(list(res_log[0]))
P_sqr_log = np.stack(list(res_log[1]))
mz_log = np.stack(list(res_log[-3]))
Pz_sqr_log = np.stack(list(res_log[-2]))
ts_log = onp.linspace(tspan_log[0], tspan_log[1], N_log + 1)

# Per-step quasi-MLE (one number per step).
sigma_sqr_per_step = onp.asarray(
    [float(quasi_mle_sigma_sqr(mz_log[i], Pz_sqr_log[i])) for i in range(N_log)]
)
# Post-hoc joint MLE.
sigma_sqr_post = float(posthoc_mle_sigma_sqr(mz_log, Pz_sqr_log))

print(f"per-step quasi-MLE: mean = {sigma_sqr_per_step.mean():.6g}")
print(f"post-hoc joint MLE:        {sigma_sqr_post:.6g}")
print(f"absolute difference:       {abs(sigma_sqr_per_step.mean() - sigma_sqr_post):.2e}")


The two numbers agree to machine precision. Now rescale the stored covariances by $\sqrt{\widehat\sigma^2}$ and compare to the uncalibrated band.

In [ ]:
P_sqr_log_cal = rescale_sqr_seq(P_sqr_log, sigma_sqr_post)
P_log = np.einsum("nij,nik->njk", P_sqr_log, P_sqr_log)
P_log_cal = np.einsum("nij,nik->njk", P_sqr_log_cal, P_sqr_log_cal)
var_uncal = onp.asarray(P_log[:, 0, 0])
var_cal = onp.asarray(P_log_cal[:, 0, 0])
x_log = onp.asarray(m_log[:, 0])

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(ts_log, x_log, "k-", lw=1.0, label="filtered mean")
ax.fill_between(
    ts_log,
    x_log - 2 * onp.sqrt(var_uncal),
    x_log + 2 * onp.sqrt(var_uncal),
    color="C0", alpha=0.25, label="uncalibrated 2-sigma",
)
ax.fill_between(
    ts_log,
    x_log - 2 * onp.sqrt(var_cal),
    x_log + 2 * onp.sqrt(var_cal),
    color="C1", alpha=0.45, label="post-hoc MLE 2-sigma",
)
ax.set_xlabel("t")
ax.set_ylabel("x(t)")
ax.set_title("logistic: uncalibrated vs post-hoc calibrated uncertainty")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


Calibration shrinks the band by the factor $\sqrt{\widehat\sigma^2}$ (here $\widehat\sigma^2 \ll 1$ because the IWP$(2)$ prior over-quotes uncertainty for this smooth problem). The shape of the band is unchanged -- post-hoc calibration is one scalar applied to every covariance.

## Part 1b -- Online (dynamic) calibration on the same problem

`ekf1_sqr_loop_dynamic` is the *online* counterpart to the post-hoc workflow above. At every fixed step it estimates the per-step quasi-MLE $\widehat\sigma^2_n = m_z^\top (H Q(h) H^\top)^{-1} m_z / d$ and bakes it into the current step's process noise *before* propagation: $P_{\text{pred}} = A P_{n-1} A^\top + \widehat\sigma^2_n\, Q_h$.

This is the **probdiffeq `MLEDiffusion`** / Bosch et al. 2021 dynamic convention: past steps keep the $\widehat\sigma^2$ that was applied when they were taken, so no global rescaling is needed. When the true diffusion is approximately constant, the per-step trace `sigma_sqr_seq` will hover near the global post-hoc estimate -- a useful sanity check.


In [ ]:
res_dyn = ekf1_sqr_loop_dynamic(
    mu0_log, S0_log, prior_log, measure_log, tspan_log, N=80
)
(
    m_seq_dyn,
    P_sqr_dyn,
    _,
    _,
    _,
    _,
    _,
    _,
    _,
    sigma_sqr_dyn,
    _,
) = res_dyn
sigma_sqr_dyn = onp.asarray(sigma_sqr_dyn)
print(
    f'global post-hoc sigma^2 = {float(sigma_sqr_post):.3e}'
    f'  |  online per-step mean = {float(sigma_sqr_dyn.mean()):.3e}'
    f'  |  spread (max/min) = {float(sigma_sqr_dyn.max()/sigma_sqr_dyn.min()):.2f}'
)

# Online band: covariances are already calibrated, no post-hoc rescale.
m_dyn = onp.stack([onp.asarray(m) for m in m_seq_dyn])
P_dyn = onp.stack(
    [onp.asarray(P_sqr.T @ P_sqr) for P_sqr in P_sqr_dyn]
)
std_dyn = onp.sqrt(P_dyn[:, 0, 0])

ts_dyn = onp.linspace(tspan_log[0], tspan_log[1], len(m_seq_dyn))
fig, ax = plt.subplots(figsize=(7, 3))
ax.fill_between(
    ts_dyn, m_dyn[:, 0] - 2 * std_dyn, m_dyn[:, 0] + 2 * std_dyn,
    alpha=0.3, label='dynamic 2-sigma'
)
ax.plot(ts_dyn, m_dyn[:, 0], label='dynamic mean')
ax.set(xlabel='t', ylabel='x(t)', title='Online dynamic-diffusion calibration')
ax.legend()
plt.tight_layout()


## Part 2 -- Adaptive step control on a multi-scale staggered problem

Now we stress-test calibration on a problem with *time-varying* scale -- two independent logistic equations with the **same growth rate** but **very different initial conditions**, giving staggered transitions:

$$
\dot x_1 = r\, x_1(1 - x_1),  \qquad x_1(0) = 10^{-5},
$$
$$
\dot x_2 = r\, x_2(1 - x_2),  \qquad x_2(0) = 10^{-10},
$$

with $r = 2$ for both. $x_1$ transitions around $t \approx 5.8$, $x_2$ around $t \approx 11.5$, and between transitions both components are near a fixed point so a well-tuned solver should take long strides. Closed-form solution: $x_i(t) = 1 / (1 + (1/x_i(0) - 1)\,e^{-r t})$.

**Why this problem stresses scalar diffusion calibration.** The two components live $5$ orders of magnitude apart in absolute value. A *scalar* $\widehat\sigma^2$ averaged across components is dominated by $x_1$'s residual; after $x_1$ has finished transitioning, both residuals are tiny and $\widehat\sigma^2 \to 0$. This **starves $x_2$ of process noise** before its own transition, so the filter overconfidently extrapolates and *misses* $x_2$'s transition altogether under the default ``calibration="dynamic"`` (the probdiffeq ``MLEDiffusion`` convention). Switching to ``calibration="cumulative"`` -- the legacy multiplicative scheme where $\widehat\sigma^2_n$ post-scales the full step covariance -- inherits the inflated $P$ from $x_1$'s transition and resolves $x_2$ correctly. We run both side-by-side.


In [ ]:
R_RATE = 2.0
X0_VEC = onp.array([1e-5, 1e-10])


def vf_dl(x, *, t):
    return np.array(
        [R_RATE * x[0] * (1 - x[0]), R_RATE * x[1] * (1 - x[1])]
    )


def analytic(t, x0, r):
    return 1.0 / (1.0 + (1.0 / x0 - 1.0) * onp.exp(-r * t))


tspan_dl = (0.0, 15.0)
q_dl = 3
prior_dl = IWP(q=q_dl, d=2)
mu0_dl, S0_dl = taylor_mode_initialization(vf_dl, np.asarray(X0_VEC), q=q_dl)
measure_dl = ODEInformation(vf_dl, prior_dl.E0, prior_dl.E1)

common_kwargs = dict(atol=1e-5, rtol=1e-3, h_min=1e-9)
res_dyn = ekf1_sqr_adaptive_loop(
    mu0_dl, S0_dl, prior_dl, measure_dl, tspan_dl,
    calibration='dynamic', **common_kwargs,
)
res_cum = ekf1_sqr_adaptive_loop(
    mu0_dl, S0_dl, prior_dl, measure_dl, tspan_dl,
    calibration='cumulative', **common_kwargs,
)
for name, r in [('dynamic', res_dyn), ('cumulative', res_cum)]:
    n_acc = len(r.h_seq)
    h_lo, h_hi = min(r.h_seq), max(r.h_seq)
    print(
        f'{name:11s}: accepted = {n_acc:3d}, rejected = {r.n_rejected:3d}, '
        f'h in [{h_lo:.3g}, {h_hi:.3g}] -- dynamic range {h_hi / h_lo:.1f}x'
    )


In [ ]:
def _components(r):
    ts = onp.asarray(r.t_seq)
    m = onp.stack([onp.asarray(mi) for mi in r.m_seq])
    x = m @ onp.asarray(prior_dl.E0.T)
    P_sqr = onp.stack([onp.asarray(P) for P in r.P_seq_sqr])
    P = onp.einsum('nij,nik->njk', P_sqr, P_sqr)
    cov_x = onp.einsum(
        'ij,njk,lk->nil', onp.asarray(prior_dl.E0), P,
        onp.asarray(prior_dl.E0)
    )
    return ts, x, cov_x[:, 0, 0], cov_x[:, 1, 1]

ts_dyn, x_dyn, var_dyn_1, var_dyn_2 = _components(res_dyn)
ts_cum, x_cum, var_cum_1, var_cum_2 = _components(res_cum)

ts_dense = onp.linspace(tspan_dl[0], tspan_dl[1], 600)
x_true_0 = analytic(ts_dense, float(X0_VEC[0]), R_RATE)
x_true_1 = analytic(ts_dense, float(X0_VEC[1]), R_RATE)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True)
for ax, idx, comp, x_true, vdyn, vcum in [
    (ax1, 0, 'x_1', x_true_0, var_dyn_1, var_cum_1),
    (ax2, 1, 'x_2', x_true_1, var_dyn_2, var_cum_2),
]:
    ax.plot(ts_dense, x_true, 'k-', lw=1.2, alpha=0.55, label='analytic')
    ax.plot(
        ts_dyn, x_dyn[:, idx], 'o-', color='C3', ms=3.0, lw=0.6,
        label=f'dynamic ({len(res_dyn.h_seq)} accepted)',
    )
    ax.plot(
        ts_cum, x_cum[:, idx], 's-', color='C2', ms=3.0, lw=0.6,
        label=f'cumulative ({len(res_cum.h_seq)} accepted)',
    )
    ax.fill_between(
        ts_cum, x_cum[:, idx] - 2 * onp.sqrt(vcum),
        x_cum[:, idx] + 2 * onp.sqrt(vcum),
        color='C2', alpha=0.25, label='cumulative 2-sigma',
    )
    ax.set_ylabel(f'${comp}(t)$')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.1, 1.2)
ax2.set_xlabel('t')
ax1.set_title(
    'Staggered multi-scale logistic: '
    'dynamic misses x_2 transition, cumulative resolves it'
)
ax1.legend(loc='center right', fontsize=8)
plt.tight_layout()
plt.show()


On $x_1$ both modes agree closely with the analytic solution. On $x_2$ they diverge dramatically: ``cumulative`` tracks the transition, ``dynamic`` collapses to zero. The root cause is the **scalar** $\widehat\sigma^2$ averaged across components -- once $x_1$ is well-tracked, the average $\widehat\sigma^2$ is small and ``dynamic`` correctly bakes that small value into $Q$, starving $x_2$. ``cumulative`` carries forward the inflated $P$ from $x_1$'s transition window, giving $x_2$ the slack it needs. The principled fix for multi-scale is per-component diffusion (probdiffeq's ``BlockDiagDiffusion``), which is not yet exposed in this library; ``cumulative`` is the pragmatic workaround.

### Phase plot

Plotting the trajectory in $(x_1, x_2)$ space reveals the staggering directly. ``cumulative`` traces the analytic L-shape: bottom-left corner $\to$ along the bottom edge as $x_1$ transitions $\to$ up the right edge as $x_2$ transitions. ``dynamic`` instead stalls in the bottom-right corner $(1, 0)$ and never climbs. Marker colour encodes $\log_{10} h$.

In [ ]:
x_true_phase_0 = analytic(ts_dense, float(X0_VEC[0]), R_RATE)
x_true_phase_1 = analytic(ts_dense, float(X0_VEC[1]), R_RATE)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.3), sharey=True)
for ax, name, r, x, ts in [
    (axes[0], 'dynamic', res_dyn, x_dyn, ts_dyn),
    (axes[1], 'cumulative', res_cum, x_cum, ts_cum),
]:
    ax.plot(
        x_true_phase_0, x_true_phase_1, 'k-', lw=1.2, alpha=0.55,
        label='analytic trajectory',
    )
    ax.plot(x[:, 0], x[:, 1], '-', color='C2', lw=0.6, alpha=0.4)
    h_arr = onp.asarray(r.h_seq)
    h_marker = onp.concatenate([[h_arr[0]], h_arr])
    log_h = onp.log10(h_marker)
    sizes = 12 + 80 * (log_h - log_h.min()) / max(
        log_h.max() - log_h.min(), 1e-12
    )
    scatter = ax.scatter(
        x[:, 0], x[:, 1],
        c=log_h, s=sizes, cmap='viridis',
        edgecolors='k', linewidths=0.4, zorder=3,
        label=f'{name} ({len(r.h_seq)} accepted)',
    )
    cb = plt.colorbar(scatter, ax=ax, shrink=0.85)
    cb.set_label(r'$\log_{10} h$')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(f"calibration='{name}'")
    ax.legend(loc='upper left', fontsize=8)
    ax.set_xlim(-0.05, 1.1)
    ax.set_ylim(-0.05, 1.1)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Left** (dynamic): the trajectory hugs the bottom edge and stalls at $(1, 0)$ -- the $x_2 \to 1$ transition never happens because $\widehat\sigma^2$ collapsed after $x_1$ finished. **Right** (cumulative): the analytic L-shape is traced correctly. Dark markers (small $h$) cluster precisely where the trajectory turns; bright markers (large $h$) sit on the smooth horizontal and vertical stretches.

### Step-size and per-step diffusion traces

Two adaptive-run diagnostics every honest report should include. The step-size dynamic range shows the *quantitative* value of adaptive control; the per-step quasi-MLE $\widehat\sigma_n^2$ tracks the same dynamics from the residual side.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.6))
t_step = ts_cum[1:]  # h_seq[k] is the step that produced ts_cum[k+1]
ax1.semilogy(t_step, res_cum.h_seq, "o-", color="C2", ms=3.5)
ax1.set_xlabel("t")
ax1.set_ylabel("h (log)")
ax1.set_title("adaptive step size")
ax1.grid(True, alpha=0.3)
ax2.semilogy(t_step, res_cum.sigma_sqr_seq, "o-", color="C3", ms=3.5)
ax2.set_xlabel("t")
ax2.set_ylabel(r"$\widehat{\sigma}^2_n$ (log)")
ax2.set_title("per-step quasi-MLE diffusion")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


$h(t)$ drops by an order of magnitude at the two transitions and recovers in between. $\widehat\sigma_n^2(t)$ spikes at the same times -- the prior covariance alone can't explain the residual at a transition, so the calibrator inflates $\sigma$. Between transitions both quantities settle to their slow-region levels.

### Whitened-residual diagnostic: Q-Q plot vs $\mathcal N(0, 1)$

**Why the pooled per-step whitened residual is *not* Gaussian.** The adaptive loop rescales each step's stored $P_z^{1/2}$ by $\sqrt{\widehat\sigma_n^2}$, so the whitened residual computed from the stored covariance is $v_n^{\text{cal}} = v_n^{\text{raw}} / \sqrt{\widehat\sigma_n^2}$ and **by construction** satisfies $\|v_n^{\text{cal}}\|^2 = d$ exactly *every* step. Pooling these forces every $d$-block of samples onto a sphere of radius $\sqrt d$ -- the distribution is *not* $\mathcal N(0, I_d)$, and a Q-Q plot of these is uninformative.

**The right diagnostic** uses a *single global* $\widehat\sigma$ to whiten -- e.g. the running mean $\widehat\sigma^2_{\text{global}} = \tfrac1N \sum_n \widehat\sigma_n^2$ (the post-hoc MLE; see Part 1). Concretely,

$$v_n^{\text{global}} \;=\; \frac{v_n^{\text{raw}}}{\sqrt{\widehat\sigma^2_{\text{global}}}} \;=\; v_n^{\text{cal}} \cdot \sqrt{\widehat\sigma_n^2 / \widehat\sigma^2_{\text{global}}}.$$

Under correct specification with truly constant $\sigma$, $\widehat\sigma_n^2 / \widehat\sigma^2_{\text{global}}$ fluctuates around 1 across steps and the pooled $v_n^{\text{global}}$ is approximately $\mathcal N(0, 1)$.

In [ ]:
sigma_sqr_n = onp.asarray(res_cum.sigma_sqr_seq)
sigma_global_sqr = float(sigma_sqr_n.mean())
print(f"sigma_global^2 (= mean of per-step sigma_n^2) = {sigma_global_sqr:.3g}")
print(
    f"sigma_n^2 spans  [{sigma_sqr_n.min():.2g}, {sigma_sqr_n.max():.2g}]  "
    f"({sigma_sqr_n.max() / sigma_sqr_n.min():.0f}x)"
)

v_global_blocks = []
for i, (mz, Pz_sqr) in enumerate(
    zip(res_cum.mz_seq, res_cum.Pz_seq_sqr, strict=True)
):
    v_stored = jax.scipy.linalg.solve_triangular(Pz_sqr.T, mz, lower=True)
    scale = onp.sqrt(sigma_sqr_n[i] / sigma_global_sqr)
    v_global_blocks.append(onp.asarray(v_stored) * scale)
z = onp.concatenate(v_global_blocks)
n = z.size
n_out = int((onp.abs(z) > 3.0).sum())
expected_out = 2 * (1 - float(jss.norm.cdf(3.0))) * n

print(
    f"n_samples = {n}, mean = {z.mean():+.3f}, std = {z.std():.3f}  "
    f"(target N(0, 1))"
)
print(f"#|z| > 3 outliers = {n_out}  (expected under N(0,1): {expected_out:.1f})")


In [ ]:
sorted_z = onp.sort(z)
p_k = (onp.arange(1, n + 1) - 0.5) / n
theoretical_q = onp.asarray(jss.norm.ppf(p_k))

fig, ax = plt.subplots(figsize=(5.8, 5.8))
lim = 4.0
ax.plot([-lim, lim], [-lim, lim], "k--", lw=1.0, alpha=0.7, label="$y = x$")
for s in (1.0, 2.0, 3.0):
    for sgn in (+1.0, -1.0):
        ax.axhline(sgn * s, color="k", lw=0.4, alpha=0.18)
        ax.axvline(sgn * s, color="k", lw=0.4, alpha=0.18)
in_box = onp.abs(sorted_z) <= lim
ax.scatter(
    theoretical_q[in_box], sorted_z[in_box],
    s=20, color="C2", edgecolors="k", linewidths=0.3,
    label=f"in-axis  ({int(in_box.sum())} / {n})",
)
if (~in_box).any():
    clipped = onp.clip(sorted_z[~in_box], -lim, lim)
    ax.scatter(
        theoretical_q[~in_box], clipped,
        s=44, marker="X", color="C3", edgecolors="k", linewidths=0.4, zorder=4,
        label=f"|z| > {lim:.0f}  ({int((~in_box).sum())}) [clipped]",
    )
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel(r"theoretical quantile  $\Phi^{-1}(p_k)$")
ax.set_ylabel("sorted global-whitened residual")
ax.set_title(
    f"Q-Q plot vs $\\mathcal{{N}}(0, 1)$  (n = {n}, mean = {z.mean():+.2f}, std = {z.std():.2f})"
)
ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()


Reading the Q-Q plot:

- The bulk of the points lies on the diagonal $y = x$ with `std(z)` close to 1 -- the constant-$\sigma$ model fits this two-scale problem reasonably well.
- A few **outliers** at the tails come from the transition steps. The per-step quasi-MLE absorbed those into a much larger $\widehat\sigma_n^2$ (visible as spikes in the previous diagnostic plot); re-whitening with the *global* $\widehat\sigma^2$ makes them stand out.

More general Q-Q failure modes:

- Line **parallel to but shifted** from the diagonal: constant mean bias (typically a one-sided EKF1 linearisation error).
- **Slope $\ne 1$**: globally mis-calibrated.
- **S-curve** through the diagonal: heavier-than-Gaussian tails (Student's-$t$ territory).
- **One-sided curl**: skewness, often a regime transition the prior cannot represent.

See the wiki note *whitened-residual-diagnostics* for the full failure-mode table.

## Where to go next

- [Diffusion Calibration](../calibration.md) -- per-step quasi-MLE, post-hoc MLE, aggregator semantics.
- [Adaptive Step-Size Control](../adaptive-steps.md) -- tolerances, the PI and P controllers, failure modes.
